## Annotate the new data's latent scores with pathways and drugs

Links each new sample's BioBombe latent scores to the same GSEA pathway
results and drug correlations used everywhere else in this project, via
`utils/pinwheels.py`'s `compute_and_plot_latent_scores` /
`assign_unique_latent_dims` — the same functions
`6.collab-data/scripts/3.pinwheel_plots.py` uses for collaborator data.

Run this script with `8.apply-biobombe-new-data/` as the working directory.

In [1]:
import json
import pathlib
import sys

import pandas as pd

sys.path.insert(0, "../utils/")
from pinwheels import assign_unique_latent_dims, compute_and_plot_latent_scores

In [2]:
data_results_dir = pathlib.Path("data/largaespada/results")

combined_latent_df = pd.read_parquet(data_results_dir / "NF1_data_latent_scores.parquet")
print(f"Latent scores: {combined_latent_df.shape}")

# Sample metadata columns (e.g. SUZ12, NF1, type) that
# 4.apply_biobombe_ensemble.py attached to combined_latent_df and recorded
# explicitly, so compute_and_plot_latent_scores below can exclude them by
# name instead of guessing from dtype or a naming pattern. Required, not
# optional: an older NF1_data_latent_scores.parquet predating this sidecar
# would otherwise silently default to an empty metadata list, letting
# SUZ12/NF1/type leak into the latent melt and fail integer conversion.
metadata_cols_file = data_results_dir / "NF1_data_latent_scores_metadata_cols.json"
if not metadata_cols_file.exists():
    raise FileNotFoundError(
        f"{metadata_cols_file} not found - this is required to know which columns "
        "in NF1_data_latent_scores.parquet are sample metadata rather than latent "
        "dimensions. Re-run 4.apply_biobombe_ensemble.py (which writes it) before "
        "this script."
    )
metadata_cols = json.loads(metadata_cols_file.read_text())
print(f"Sample metadata columns to exclude from latent melting: {metadata_cols}")

Latent scores: (3888, 408)
Sample metadata columns to exclude from latent melting: ['SUZ12', 'NF1', 'type']


### Pathway annotation: CORUM and Reactome, kept separate

A real, ensemble-wide Reactome run did happen for this project at some
point — 4.drug-dependency/results/all_reactome_results.parquet holds genuine
Reactome pathways (R-HSA-tagged names, e.g. "rRNA Modification In Nucleus
And Cytosol R-HSA-6790901") spanning all 6 model types, so this isn't
hypothetical. What's missing is the raw, per-latent-dimension GSEA hit table
that would have fed it (the direct analog of combined_z_matrix_gsea_results.parquet,
but for Reactome) — it isn't in 3.run-biobombe/gsea_results/ or anywhere
else in this repo's git history. `reactome_gsea_path` below is where
`2.interpret_biobombe_with_gsea.py` would need to write one (by re-running
it with a Reactome library) for that half to populate again.

`3.run-biobombe/gsea_results/combined_z_matrix_gsea_results.parquet` — the
one raw file that does exist — is CORUM, confirmed directly: every one of
its 170 pathway names matches CORUM's complex-naming convention ("X complex
(human)"), never Reactome's R-HSA- IDs. `4.drug-dependency/scripts/7.pinwheel_plots.py`
actually has this backwards (it labels this file `reactome_df` and expects
a nonexistent "_corum"-suffixed file for CORUM) — that script has never
successfully run against its current code, since the file it wants for
CORUM was never committed either.

Both results, when present, use a column literally called "reactome_pathway"
for the pathway name — that's `2.interpret_biobombe_with_gsea.py`'s schema
regardless of which library was actually run, not a labeling mistake here.

### Pathway annotation (CORUM GSEA results)

In [3]:
gsea_sources = {
    "CORUM": pathlib.Path("../3.run-biobombe/gsea_results/combined_z_matrix_gsea_results.parquet"),
    "Reactome": pathlib.Path("../3.run-biobombe/gsea_results/combined_z_matrix_gsea_results_reactome.parquet"),
}

pathway_max_by_library = {}
for library, gsea_path in gsea_sources.items():
    if not gsea_path.exists():
        print(f"{library}: {gsea_path} not found, skipping.")
        continue

    gsea_df = pd.read_parquet(gsea_path)
    # Drop the shuffled negative-control rows and keep only nominally significant
    # hits. This file has no fdr column (unlike the single-VAE GSEA results
    # elsewhere in the project), so significance here is by p_value alone.
    significant_gsea_df = gsea_df[(~gsea_df["shuffled"]) & (gsea_df["p_value"] < 0.05)].copy()
    print(f"{library}: {significant_gsea_df.shape[0]} / {gsea_df.shape[0]} significant hits")

    # nes_score (normalized enrichment score), not gsea_es_score: NES adjusts
    # for gene-set-size effects, and it's what the previously-run ensemble-wide
    # annotation (4.drug-dependency/results/all_corum_results.parquet /
    # all_reactome_results.parquet) actually used.
    pathway_max_by_library[library] = assign_unique_latent_dims(
        significant_gsea_df, score_col="nes_score", target_col="reactome_pathway"
    )

CORUM: 52386 / 1142910 significant hits
Processing pathway: C complex spliceosome (human)
Assigned latent dimension 2 for model vanillavae, 18, init 2 to pathway C complex spliceosome (human).
Processing pathway: Mediator complex (human)
Assigned latent dimension 2 for model betavae, 10, init 1 to pathway Mediator complex (human).
Processing pathway: Spliceosome (human)
Assigned latent dimension 2 for model betavae, 3, init 0 to pathway Spliceosome (human).
Processing pathway: DAB complex (human)
Assigned latent dimension 5 for model betatcvae, 12, init 1 to pathway DAB complex (human).
Processing pathway: DA complex (human)


Assigned latent dimension 45 for model nmf, 200, init 0 to pathway DA complex (human).
Processing pathway: BRM-SIN3A-HDAC complex (human)
Assigned latent dimension 2 for model betatcvae, 2, init 0 to pathway BRM-SIN3A-HDAC complex (human).
Processing pathway: STAGA complex (SPT3-TAF9-GCN5 acetyltransferase complex) (human)
Assigned latent dimension 40 for model betatcvae, 50, init 1 to pathway STAGA complex (SPT3-TAF9-GCN5 acetyltransferase complex) (human).
Processing pathway: Polycystin-1 multiprotein complex (ACTN1, CDH1, SRC, JUP, VCL, CTNNB1, PXN, BCAR1, PKD1, PTK2, TLN1) (human)
Assigned latent dimension 35 for model betatcvae, 80, init 4 to pathway Polycystin-1 multiprotein complex (ACTN1, CDH1, SRC, JUP, VCL, CTNNB1, PXN, BCAR1, PKD1, PTK2, TLN1) (human).
Processing pathway: mRNA decay complex (UPF1, UPF2, UPF3B, DCP2, XRN1, XRN2, EXOSC2, EXOSC4, EXOSC10, PARN) (human)
Assigned latent dimension 5 for model betatcvae, 16, init 4 to pathway mRNA decay complex (UPF1, UPF2, UPF3B, 

Assigned latent dimension 4 for model betatcvae, 9, init 4 to pathway Nup 107-160 subcomplex (human).
Processing pathway: DRIP complex (human)
Assigned latent dimension 16 for model betatcvae, 200, init 4 to pathway DRIP complex (human).
Processing pathway: BAF complex (human)
Assigned latent dimension 100 for model betatcvae, 150, init 3 to pathway BAF complex (human).
Processing pathway: ARC-L complex (human)
Assigned latent dimension 3 for model betatcvae, 10, init 2 to pathway ARC-L complex (human).
Processing pathway: cMYC-ATPase-helicase complex (human)
Assigned latent dimension 9 for model betatcvae, 18, init 3 to pathway cMYC-ATPase-helicase complex (human).
Processing pathway: Emerin complex 52 (human)
Assigned latent dimension 60 for model betatcvae, 90, init 4 to pathway Emerin complex 52 (human).
Processing pathway: BLM complex II (human)
Assigned latent dimension 14 for model betatcvae, 14, init 1 to pathway BLM complex II (human).
Processing pathway: Cytochrome bc1-comple

Assigned latent dimension 5 for model nmf, 9, init 0 to pathway Respiratory chain complex I (early intermediate NDUFAF1 assembly), mitochondrial (human).
Processing pathway: Kaiso-NCOR complex (human)
Assigned latent dimension 6 for model betatcvae, 25, init 0 to pathway Kaiso-NCOR complex (human).
Processing pathway: Kinase maturation complex 1 (human)
Assigned latent dimension 50 for model pca, 60, init 0 to pathway Kinase maturation complex 1 (human).
Processing pathway: RNA polymerase II holoenzyme complex (human)
Assigned latent dimension 3 for model betatcvae, 18, init 1 to pathway RNA polymerase II holoenzyme complex (human).
Processing pathway: Large Drosha complex (human)
Assigned latent dimension 25 for model betatcvae, 60, init 1 to pathway Large Drosha complex (human).
Processing pathway: PYR complex (mouse)
Assigned latent dimension 90 for model betavae, 200, init 3 to pathway PYR complex (mouse).
Processing pathway: SNF2h-cohesin-NuRD complex (human)
Assigned latent dimen

In [4]:
pathway_results = []
for library, pathway_max in pathway_max_by_library.items():
    for model_id in combined_latent_df["ModelID"].unique():
        df = compute_and_plot_latent_scores(
            model_id, combined_latent_df, pathway_max, "reactome_pathway", "nes_score",
            library, make_plot=False, metadata_cols=metadata_cols,
        )
        df["gene_set_library"] = library
        pathway_results.append(df)

pathway_results_df = pd.concat(pathway_results, ignore_index=True) if pathway_results else pd.DataFrame()
print(f"Pathway annotations (CORUM + Reactome combined): {pathway_results_df.shape}")

S462
ST88-14
STS-26T


iHSC1L_C3
iHSC1L_F12
iPN1L


iPN2J
iPN3D
Pathway annotations (CORUM + Reactome combined): (1235, 11)


### Drug annotation

This step needs `4.drug-dependency/results/combined_latent_drug_correlations.parquet`,
the ensemble-wide drug correlation table `4.drug-dependency/scripts/5.drug_latent_dim_correlation.py`
produces. That script depends on `saved_models/`, which only exists locally
now — if it hasn't been run yet, drug annotation is skipped here rather than failing.

In [5]:
drug_correlation_path = pathlib.Path("../4.drug-dependency/results/combined_latent_drug_correlations.parquet")

if drug_correlation_path.exists():
    drug_df = pd.read_parquet(drug_correlation_path)
    significant_drug_df = drug_df[
        (~drug_df["shuffled"]) & (drug_df["pearson_correlation"].abs() > 0.15) & (drug_df["p_value"] < 0.05)
    ].copy()
    drug_max = assign_unique_latent_dims(
        significant_drug_df, score_col="pearson_correlation", target_col="name"
    )

    drug_results = []
    for model_id in combined_latent_df["ModelID"].unique():
        df = compute_and_plot_latent_scores(
            model_id, combined_latent_df, drug_max, "name", "pearson_correlation", "Drug",
            make_plot=False, metadata_cols=metadata_cols,
        )
        drug_results.append(df)
    drug_results_df = pd.concat(drug_results, ignore_index=True) if drug_results else pd.DataFrame()
    print(f"Drug annotations: {drug_results_df.shape}")
else:
    drug_results_df = pd.DataFrame()
    print(
        f"{drug_correlation_path} not found — skipping drug annotation. "
        "Run 4.drug-dependency/scripts/5.drug_latent_dim_correlation.py first to enable it."
    )

Processing pathway: idasanutlin
Assigned latent dimension 4 for model nmf, 25, init 0 to pathway idasanutlin.
Processing pathway: AMG-232
Assigned latent dimension 8 for model ica, 10, init 0 to pathway AMG-232.
Processing pathway: CGM097
Assigned latent dimension 29 for model nmf, 45, init 0 to pathway CGM097.
Processing pathway: XL-647
Assigned latent dimension 9 for model nmf, 50, init 0 to pathway XL-647.
Processing pathway: nutlin-3
Assigned latent dimension 29 for model nmf, 30, init 0 to pathway nutlin-3.
Processing pathway: poziotinib
Assigned latent dimension 10 for model nmf, 18, init 0 to pathway poziotinib.
Processing pathway: atiprimod
Assigned latent dimension 72 for model ica, 100, init 0 to pathway atiprimod.
Processing pathway: ibrutinib
Assigned latent dimension 6 for model nmf, 12, init 0 to pathway ibrutinib.
Processing pathway: AZ-628


Assigned latent dimension 16 for model nmf, 60, init 0 to pathway AZ-628.
Processing pathway: osimertinib
Assigned latent dimension 6 for model nmf, 16, init 0 to pathway osimertinib.
Processing pathway: LGX818
Assigned latent dimension 19 for model nmf, 100, init 0 to pathway LGX818.
Processing pathway: tyrphostin-AG-1478
Assigned latent dimension 6 for model nmf, 25, init 0 to pathway tyrphostin-AG-1478.
Processing pathway: neratinib
Assigned latent dimension 9 for model nmf, 60, init 0 to pathway neratinib.
Processing pathway: vemurafenib
Assigned latent dimension 16 for model ica, 45, init 0 to pathway vemurafenib.
Processing pathway: OSI-420
Assigned latent dimension 6 for model nmf, 70, init 0 to pathway OSI-420.
Processing pathway: PD-153035
Assigned latent dimension 6 for model nmf, 14, init 0 to pathway PD-153035.
Processing pathway: canertinib
Assigned latent dimension 7 for model ica, 50, init 0 to pathway canertinib.
Processing pathway: bafetinib
Assigned latent dimension 5

Assigned latent dimension 0 for model nmf, 7, init 0 to pathway ARRY-334543.
Processing pathway: AEE788
Assigned latent dimension 67 for model nmf, 80, init 0 to pathway AEE788.
Processing pathway: BMS-599626
Assigned latent dimension 29 for model ica, 30, init 0 to pathway BMS-599626.
Processing pathway: lapatinib
Assigned latent dimension 27 for model ica, 70, init 0 to pathway lapatinib.
Processing pathway: taselisib
Assigned latent dimension 12 for model ica, 18, init 0 to pathway taselisib.
Processing pathway: GDC-0879
Assigned latent dimension 13 for model nmf, 80, init 0 to pathway GDC-0879.
Processing pathway: PD-168393
Assigned latent dimension 24 for model nmf, 35, init 0 to pathway PD-168393.
Processing pathway: BIBX-1382
Assigned latent dimension 93 for model nmf, 100, init 0 to pathway BIBX-1382.
Processing pathway: PD-0325901
Assigned latent dimension 8 for model ica, 14, init 0 to pathway PD-0325901.
Processing pathway: MEK162
Assigned latent dimension 28 for model vanil

Assigned latent dimension 16 for model nmf, 50, init 0 to pathway SB-590885.
Processing pathway: AS-703026
Assigned latent dimension 0 for model ica, 10, init 0 to pathway AS-703026.
Processing pathway: oxaliplatin
Assigned latent dimension 80 for model betatcvae, 200, init 3 to pathway oxaliplatin.
Processing pathway: BVD-523
Assigned latent dimension 5 for model betatcvae, 30, init 4 to pathway BVD-523.
Processing pathway: EVP4593
Assigned latent dimension 71 for model betatcvae, 200, init 4 to pathway EVP4593.
Processing pathway: selumetinib


Assigned latent dimension 22 for model ica, 50, init 0 to pathway selumetinib.
Processing pathway: refametinib
Assigned latent dimension 18 for model ica, 30, init 0 to pathway refametinib.
Processing pathway: NVP-BHG712
Assigned latent dimension 16 for model ica, 35, init 0 to pathway NVP-BHG712.
Processing pathway: UNC2250
Assigned latent dimension 12 for model betatcvae, 30, init 0 to pathway UNC2250.
Processing pathway: U-0126
Assigned latent dimension 13 for model nmf, 45, init 0 to pathway U-0126.
Processing pathway: cobimetinib
Assigned latent dimension 5 for model betatcvae, 35, init 3 to pathway cobimetinib.
Processing pathway: chlormidazole
Assigned latent dimension 69 for model betatcvae, 70, init 1 to pathway chlormidazole.
Processing pathway: verubulin
Assigned latent dimension 129 for model betatcvae, 200, init 0 to pathway verubulin.
Processing pathway: afatinib
Assigned latent dimension 0 for model nmf, 8, init 0 to pathway afatinib.
Processing pathway: PF-03814735


Assigned latent dimension 101 for model betatcvae, 150, init 3 to pathway PF-03814735.
Processing pathway: trametinib
Assigned latent dimension 4 for model nmf, 6, init 0 to pathway trametinib.
Processing pathway: PLX-4720
Assigned latent dimension 13 for model nmf, 40, init 0 to pathway PLX-4720.
Processing pathway: WZ8040
Assigned latent dimension 6 for model nmf, 35, init 0 to pathway WZ8040.
Processing pathway: RKI-1447
Assigned latent dimension 34 for model betatcvae, 90, init 0 to pathway RKI-1447.
Processing pathway: erlotinib
Assigned latent dimension 88 for model nmf, 90, init 0 to pathway erlotinib.
Processing pathway: PD-198306
Assigned latent dimension 73 for model nmf, 200, init 0 to pathway PD-198306.
Processing pathway: mebendazole
Assigned latent dimension 118 for model betatcvae, 150, init 3 to pathway mebendazole.
Processing pathway: TAK-632
Assigned latent dimension 32 for model nmf, 40, init 0 to pathway TAK-632.
Processing pathway: AZD8330
Assigned latent dimension

... (68 outputs left out)

No unused latent dimensions left for pathway aloperine.
Processing pathway: bentiromide
Assigned latent dimension 61 for model ica, 100, init 0 to pathway bentiromide.
Processing pathway: ursolic-acid
No unused latent dimensions left for pathway ursolic-acid.
Processing pathway: trioxsalen
Assigned latent dimension 16 for model pca, 20, init 0 to pathway trioxsalen.
Processing pathway: valethamate
No unused latent dimensions left for pathway valethamate.
Processing pathway: carprofen
Assigned latent dimension 15 for model nmf, 200, init 0 to pathway carprofen.
Processing pathway: isoleucine
No unused latent dimensions left for pathway isoleucine.
Processing pathway: brexpiprazole
No unused latent dimensions left for pathway brexpiprazole.
Processing pathway: milnacipran
Assigned latent dimension 3 for model vanillavae, 90, init 1 to pathway milnacipran.
Processing pathway: gaboxadol
No unused latent dimensions left for pathway gaboxadol.
Processing pathway: butalbital
No unused latent 

S462
ST88-14
STS-26T


iHSC1L_C3
iHSC1L_F12
iPN1L


iPN2J
iPN3D
Drug annotations: (31128, 10)


### Save top associations per sample

In [6]:
top_pathways_per_sample = (
    pathway_results_df.sort_values(["ModelID", "gene_set_library", "pathway_score"], ascending=[True, True, False])
    .groupby(["ModelID", "gene_set_library"])
    .head(20)
    if not pathway_results_df.empty else pathway_results_df
)
top_pathways_per_sample.to_parquet(data_results_dir / "NF1_data_top_pathways.parquet", index=False)

if not drug_results_df.empty:
    top_drugs_per_sample = (
        drug_results_df.sort_values(["ModelID", "pathway_score"], ascending=[True, False])
        .groupby("ModelID")
        .head(20)
    )
    top_drugs_per_sample.to_parquet(data_results_dir / "NF1_data_top_drugs.parquet", index=False)

print(f"Saved annotated results to {data_results_dir}")

Saved annotated results to data/largaespada/results
